In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
arashnic_har_1_path = kagglehub.dataset_download('arashnic/har-1')

print('Data source import complete.')


Using Colab cache for faster access to the 'har-1' dataset.
Data source import complete.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/har-1/Time_domain_subsamples/3.Time_domain_subsamples/KU-HAR_v1.0_raw_samples.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1052_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1017_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1014_C_6.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1042_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1021_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1073_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1034_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1023_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1018_C_1.csv
/kaggle/input/har-1/Raw_time_domian_data/1.Raw_time_domian_data/2.Talk-sit/1001_C_1.csv
/kaggle/input/har-1/Raw_

In [ ]:
# IMPORTANT: RUN THIS CELL TO INSTALL TENSORFLOW
!pip install tensorflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, LSTM,
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# Seed for reproducibility — same seed = same results each run
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
import os
CSV_PATH = os.path.join(arashnic_har_1_path, 'Time_domain_subsamples', '3.Time_domain_subsamples', 'KU-HAR_v1.0_raw_samples.csv')

df = pd.read_csv(CSV_PATH)

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 3 rows:')
df.head(3)

Shape: (9184, 9003)

Columns: ['-0.019247', '0.018654', '-0.0005703', '0.013523', '0.013277', '0.022462', '-0.0014467', '0.01748', '0.031381', '0.0049953', '0.0025644', '0.00015259', '0.00012398', '0.011853', '0.0092697', '0.01381', '0.018312', '0.027507', '0.0012627', '-0.0080137', '0.022958', '0.036942', '0.036606', '0.043349', '0.061793', '0.037542', '0.027842', '0.0041666', '-0.01439', '-0.0256', '-0.029591', '-0.035906', '-0.042069', '-0.010467', '0.0041008', '0.039618', '0.013264', '0.0062017', '-0.0054464', '-0.031066', '0.00026703', '0.0098801', '0.014614', '0.00040913', '0.0099754', '0.012335', '-0.011252', '-0.022682', '-0.031631', '-0.04048', '-0.018608', '-0.0089064', '0.012411', '0.031013', '0.049309', '0.048456', '0.017001', '-0.0068264', '-0.030251', '-0.029755', '-0.050543', '-0.06397', '-0.034812', '-0.036751', '0.0084686', '0.019977', '0.031308', '0.037766', '0.018215', '0.0037022', '0.020058', '0.010192', '-0.0042343', '-0.013711', '-0.0018444', '0.01455', '-0.004662

,-0.019247,0.018654,-0.0005703,0.013523,0.013277,0.022462,-0.0014467,0.01748,0.031381,0.0049953,...,0.3041,0.3042,0.3043,0.3044,0.3045,0.3046,0.3047,0.3048,992,1
0,0.009649,0.004752,-0.014206,-0.018720,-0.027882,-0.041592,-0.007939,-0.012533,-0.002917,0.027736,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,992,2
1,-0.028546,-0.018661,0.005173,0.012134,0.026038,0.025596,0.001604,0.013366,0.013118,-0.017758,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,992,3
2,-0.019654,-0.033483,-0.011746,-0.016304,-0.016103,0.014754,0.009741,-0.002241,-0.009273,-0.016154,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,992,4


In [ ]:
X_raw = df.iloc[:, :9000].values   # shape: (9185, 9000)

# y: column at index 9000 — the activity class label
y = df.iloc[:, 9000].values.astype('int32')   # shape: (9185,)

# (Optional) Store actual signal length for masking later
lengths = df.iloc[:, 9001].values

print("X_raw shape:", X_raw.shape)
print("y shape    :", y.shape)
print("Unique classes:", np.unique(y))
print("Label dtype:", y.dtype)

X_raw shape: (9184, 9000)
y shape    : (9184,)
Unique classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
Label dtype: int32


In [ ]:
#  Cell 4 — Reshape (9185, 9000) → (9185, 1500, 6)

N_SAMPLES   = X_raw.shape[0]   # 9185
N_TIMESTEPS = 1500              # each channel has 1500 time points
N_CHANNELS  = 6                # ax, ay, az, gx, gy, gz

X = X_raw.reshape(N_SAMPLES, N_TIMESTEPS, N_CHANNELS)

print("Before reshape:", X_raw.shape)
print("After  reshape:", X.shape)

# Sanity check — verify first sample, first timestep, all 6 channels
print("\nSample 0, timestep 0 (all 6 sensors):")
print(X[0, 0, :])

# Check zero-padding at the end of a padded sample
print("\nSample 0, last 5 timesteps (may be zeros if padded):")
print(X[0, -5:, :])

Before reshape: (9184, 9000)
After  reshape: (9184, 1500, 6)

Sample 0, timestep 0 (all 6 sensors):
[ 0.0096493  0.0047522 -0.014206  -0.01872   -0.027882  -0.041592 ]

Sample 0, last 5 timesteps (may be zeros if padded):
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


In [ ]:
#  Cell 5 — Normalize per channel using StandardScaler
# We temporarily flatten to 2D to fit the scaler, then reshape back

# Step 1: Flatten (9185, 1500, 6) → (9185 * 1500, 6)
# so scaler sees each channel as a column across ALL timesteps
X_flat = X.reshape(-1, N_CHANNELS)   # shape: (13,777,500, 6)

scaler = StandardScaler()

# FIT only on X_flat (we'll do train/test split AFTER scaling here
# OR we can split first then scale — see note below)
# For simplicity, we scale first then split.
# Advanced: split first to prevent any leakage.
X_flat_scaled = scaler.fit_transform(X_flat)

# Step 2: Reshape back to 3D
X_scaled = X_flat_scaled.reshape(N_SAMPLES, N_TIMESTEPS, N_CHANNELS)

print("Scaled X shape:", X_scaled.shape)
print("Channel 0 mean (should ≈ 0):", X_scaled[:, :, 0].mean().round(4))
print("Channel 0 std  (should ≈ 1):", X_scaled[:, :, 0].std().round(4))


Scaled X shape: (9184, 1500, 6)
Channel 0 mean (should ≈ 0): 0.0
Channel 0 std  (should ≈ 1): 1.0


In [ ]:
# Cell 6 — Stratified 80/20 train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y        # preserve class distribution
)

print("X_train shape:", X_train.shape)   # (7348, 1500, 6)
print("X_test  shape:", X_test.shape)    # (1837, 1500, 6)
print("y_train shape:", y_train.shape)
print("y_test  shape:", y_test.shape)

# Verify class proportions are preserved
print("\nClass distribution in train:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

X_train shape: (7347, 1500, 6)
X_test  shape: (1837, 1500, 6)
y_train shape: (7347,)
y_test  shape: (1837,)

Class distribution in train:
{np.int32(0): np.int64(476), np.int32(1): np.int64(477), np.int32(2): np.int64(463), np.int32(3): np.int64(475), np.int32(4): np.int64(633), np.int32(5): np.int64(470), np.int32(6): np.int64(434), np.int32(7): np.int64(507), np.int32(8): np.int64(442), np.int32(9): np.int64(410), np.int32(10): np.int64(331), np.int32(11): np.int64(463), np.int32(12): np.int64(174), np.int32(13): np.int64(138), np.int32(14): np.int64(404), np.int32(15): np.int64(406), np.int32(16): np.int64(426), np.int32(17): np.int64(218)}


In [ ]:
# Cell 7 — Define the CNN + LSTM architecture

N_CLASSES = 18   # classes 0 through 17

model = Sequential([

    # --- CNN Block 1: local feature extraction ---
    Conv1D(filters=64, kernel_size=3, activation='relu',
           input_shape=(N_TIMESTEPS, N_CHANNELS)),
    # kernel_size=3 means we look at 3 consecutive timesteps at once
    BatchNormalization(),   # stabilizes activations, speeds up training
    MaxPooling1D(pool_size=2),   # halves the sequence length

    # --- CNN Block 2: higher-level patterns ---
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    # --- LSTM: temporal context across the compressed sequence ---
    LSTM(units=128, return_sequences=False),
    # return_sequences=False → output only the final hidden state
    # (we want one prediction per sample, not one per timestep)

    # --- Classification head ---
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(N_CLASSES, activation='softmax')
    # softmax: outputs 18 probabilities that sum to 1.0
])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1498, 64)       │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1498, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 749, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 747, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 747, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 373, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,698 (655.07 KB)

 Trainable params: 167,314 (653.57 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# Cell 8 — Compile the model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    # Use this when y labels are raw integers (0, 1, 2 ... 17)
    # NOT one-hot encoded
    metrics=['accuracy']
)

print("Model compiled successfully.")
print("Total trainable parameters:",
      model.count_params())

Model compiled successfully.
Total trainable parameters: 167698


In [ ]:
# Cell 9 — Train the model with early stopping

early_stop = EarlyStopping(
    monitor='val_loss',     # watch validation loss
    patience=10,             # wait 10 epochs before stopping
    restore_best_weights=True  # revert to best weights
)

history = model.fit(
    X_train, y_train,
    epochs=50,               # max epochs (early stop may cut short)
    batch_size=32,
    validation_split=0.1,   # 10% of train data for validation
    callbacks=[early_stop],
    verbose=1               # print progress bar
)

print("\nTraining complete!")
print("Stopped at epoch:", len(history.history['loss']))


Epoch 1/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 199s 901ms/step - accuracy: 0.0829 - loss: 2.8354 - val_accuracy: 0.1238 - val_loss: 3.1683
Epoch 2/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 182s 880ms/step - accuracy: 0.2482 - loss: 2.1014 - val_accuracy: 0.2966 - val_loss: 2.0505
Epoch 3/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 182s 878ms/step - accuracy: 0.3309 - loss: 1.7273 - val_accuracy: 0.5156 - val_loss: 1.3498
Epoch 4/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 181s 876ms/step - accuracy: 0.3274 - loss: 1.8115 - val_accuracy: 0.4245 - val_loss: 1.4393
Epoch 5/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 202s 878ms/step - accuracy: 0.4114 - loss: 1.4768 - val_accuracy: 0.5252 - val_loss: 1.1659
Epoch 6/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 182s 882ms/step - accuracy: 0.4876 - loss: 1.2613 - val_accuracy: 0.5524 - val_loss: 1.0927
Epoch 7/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 183s 884ms/step - accuracy: 0.5327 - loss: 1.1725 - val_accuracy: 0.6109 - val_loss: 0.9491
Epoch 8/50
207/207 ━━━━━━━━━━━━━━━━━━━━ 202s 882ms/step - accuracy: 0.5694 -

In [ ]:
# Cell 10 — Full evaluation on test set

# Step 1: Overall loss and accuracy
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy : {test_acc * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

# Step 2: Per-class predictions
y_pred_probs = model.predict(X_test)          # shape: (1837, 18)
y_pred = np.argmax(y_pred_probs, axis=1)      # pick class with highest prob

# Step 3: Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Step 4: Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'C{i}' for i in range(18)],
            yticklabels=[f'C{i}' for i in range(18)])
plt.title('Confusion Matrix — KU-HAR CNN+LSTM')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# Cell 11 — Plot training and validation curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
axes[0].plot(history.history['accuracy'],
             label='Train Accuracy', color='steelblue', lw=2)
axes[0].plot(history.history['val_accuracy'],
             label='Val Accuracy', color='darkorange', lw=2, linestyle='--')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curves
axes[1].plot(history.history['loss'],
             label='Train Loss', color='steelblue', lw=2)
axes[1].plot(history.history['val_loss'],
             label='Val Loss', color='darkorange', lw=2, linestyle='--')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

# Quick overfitting check
final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
gap = final_train_acc - final_val_acc
print(f"Train-Val accuracy gap: {gap * 100:.1f}%")
if gap > 0.05:
    print("⚠ Gap > 5% — consider increasing Dropout or reducing model size")
else:
    print("✓ Gap is acceptable — model generalizes well")
